# Generate SQL Insert Statements
### Import necessary packages

In [5]:
import pandas as pd
import os

In [6]:
# Read the cleaned csv files
movies = pd.read_csv("../data/clean/movies_clean.csv")
boxoffice_df = pd.read_csv("../data/clean/boxoffice_clean.csv")
rotten_df = pd.read_csv("../data/clean/rotten_clean.csv")

### Create `create_tables.sql` file

**Description of `create_tables.sql` file:**

box_id NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY, and review_id NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY, in their corresponding tables were indicated in order to store numeric values.
- `GENERATED ALWAYS AS IDENTITY` allows us to make the database automatically assign a new incrementing value every time a row is inserted, also inhibiting manual insertion of the id values
- `PRIMARY KEY` enforces that every value in this column is unique
- every time a new review is inserted into the tables, it will be assigned the next available id value

`VARCHAR2` is used because it stores strings that vary in length.
- ex. if a movie title is 10 characters long, it only stores the 10 characters
- this data type is used for strings such as `norm_title` and `critics_consensus`, as the length varies for each row

`CHAR` is used to store `tconst` values, as each tconst value is always exactly 10 characters

In [7]:
create_sql = """
-- Drop existing tables 
DROP TABLE rt_reviews PURGE;
DROP TABLE box_office PURGE;
DROP TABLE movie_genres PURGE;
DROP TABLE movies PURGE;

PURGE RECYCLEBIN;

-- Create movies table
CREATE TABLE movies (
    tconst CHAR(10) PRIMARY KEY,
    norm_title VARCHAR2(300) NOT NULL,
    start_year INTEGER NOT NULL,
    runtime_minutes INTEGER,
    imdb_rating FLOAT);

-- Create movie genres table 
CREATE TABLE movie_genres (
    tconst CHAR(10) NOT NULL,
    genre VARCHAR2(100),
    PRIMARY KEY (tconst, genre),
    FOREIGN KEY (tconst) REFERENCES movies(tconst) ON DELETE CASCADE);

-- Create box office table
CREATE TABLE box_office (
    box_id NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    norm_title VARCHAR2(300),
    year INTEGER,
    worldwide_revenue FLOAT);

-- Create rotten tomato reviews table 
CREATE TABLE rt_reviews (
    review_id NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    norm_title VARCHAR2(300),
    tomatometer_rating INTEGER,
    critics_consensus VARCHAR2(5000));
"""

with open("../sql/create_tables.sql", "w") as f: # Opens and writes SQL to file, creating it if it doesn't exist
    f.write(create_sql)

### Create helper functions to handle NULL values when building INSERT statements

In [8]:
def safe_int(value):
    """
    Convert numeric string to int or return NULL if missing.
    """
    if pd.isna(value) or value == "\\N":
        return "NULL"
    return int(value)

def safe_float(value):
    """
    Convert numeric string to float or return NULL if missing.
    """
    if pd.isna(value) or value == "\\N":
        return "NULL"
    return float(value)

def safe_str(value):
    """
    Returns a safe string for SQL parsing, or NULL if value is missing.
    Single quotes in a string are doubled to avoid confusion of where string ends.
    ex. "Schindler's List" becomes "'Schindler''s List'"
    """
    if pd.isna(value):
        return "NULL"
    return "'" + str(value).replace("'", "''") + "'"


### Insert values into `movies` table

In [9]:
sql_lines = []

for _, row in movies.iterrows():
    tconst      = safe_str(row["tconst"])
    norm_title  = safe_str(row["norm_title"])
    start_year  = safe_int(row["startYear"])
    runtime     = safe_int(row["runtimeMinutes"])
    imdb_rating = safe_float(row["averageRating"])

    sql = f"""INSERT INTO movies (tconst, norm_title, start_year, runtime_minutes, imdb_rating) 
                VALUES ({tconst}, {norm_title}, {start_year}, {runtime}, {imdb_rating});"""
    sql_lines.append(sql)

sql_lines.append("COMMIT;")

with open("../sql/movies_inserts.sql", "w", encoding = "utf-8") as f: # Opens and writes SQL to file, saving it to movies_inserts.sql
    f.write("\n".join(sql_lines))

print("movies_inserts.sql generated")

movies_inserts.sql generated


### Insert values into `movie_genres` table

In [10]:
sql_lines = []

for _, row in movies.iterrows():
    if pd.isna(row["genres"]): # If there are NA values in genre column, continue parsing
        continue
    for genre in row["genres"].split(","):
        genre_clean = safe_str(genre.strip())
        tconst = safe_str(row["tconst"])
        sql = f"""INSERT INTO movie_genres (tconst, genre) 
                    VALUES ({tconst}, {genre_clean});"""
        sql_lines.append(sql)

sql_lines.append("COMMIT;")

with open("../sql/movies_genres_inserts.sql", "w", encoding="utf-8") as f: # Opens and writes SQL to file, saving it to movies_genres_inserts.sql
    f.write("\n".join(sql_lines))

print("movies_genres_inserts.sql generated")

movies_genres_inserts.sql generated


### Insert values into `box_office` table

In [11]:
sql_lines = ["SET DEFINE OFF;"] # Prevents "&" from being executed

for _, row in boxoffice_df.iterrows():
    norm_title = safe_str(row["norm_title"])
    year       = safe_int(row["Year"])
    worldwide  = safe_float(row["$Worldwide"])

    sql = f"""INSERT INTO box_office (norm_title, year, worldwide_revenue) 
                VALUES ({norm_title}, {year}, {worldwide});"""
    sql_lines.append(sql)

sql_lines.append("COMMIT;")

with open("../sql/boxoffice_inserts.sql", "w", encoding="utf-8") as f: # Opens and writes SQL to file, saving it to boxoffice_inserts.sql
    f.write("\n".join(sql_lines))

print("boxoffice_inserts.sql generated")

boxoffice_inserts.sql generated


### Insert values into `rt_reviews` table

In [12]:
ssql_lines = ["SET DEFINE OFF;"] # This prevents the "&" character from being executed

for _, row in rotten_df.iterrows():
    norm_title  = safe_str(row["norm_title"])
    tomatometer = safe_int(row["tomatometer_rating"])
    consensus   = safe_str(row["critics_consensus"])

    sql = f"""INSERT INTO rt_reviews (norm_title, tomatometer_rating, critics_consensus) 
                VALUES ({norm_title}, {tomatometer}, {consensus});"""
    sql_lines.append(sql)

sql_lines.append("COMMIT;")

with open("../sql/rt_reviews_inserts.sql", "w", encoding="utf-8") as f: # Opens and writes SQL to file, saving it to rt_reviews_inserts.sql
    f.write("\n".join(sql_lines))

print("rt_reviews_inserts.sql generated")

rt_reviews_inserts.sql generated
